### Importing the data

In [18]:
! curl "https://api.mockaroo.com/api/04e663c0?count=1000&key=c225bff0" > "department.csv"
! curl "https://api.mockaroo.com/api/36571b90?count=1000&key=c225bff0" > "Employees.csv"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100 40662    0 40662    0     0  20718      0 --:--:--  0:00:01 --:--:-- 20745
100 57011    0 57011    0     0  25961      0 --:--:--  0:00:02 --:--:-- 26008
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  5973    0  5973    0     0   5972      0 --:--:--  0:00:01 --:--:--  5984
100 81889    0 81889    0     0  48429      0 --:--:--  0:00:01 --:--:-- 48483
100 93496    0 93496    0     0  52082      0 --:--:--  0:00:01 --:--:-- 52145


### Setting the Database

In [19]:
import os
import sqlite3
import pandas as pd

In [20]:
department_schema = """
CREATE TABLE IF NOT EXISTS department (
	department_id INT PRIMARY KEY,
	department_name VARCHAR(50),
	department_code VARCHAR(9),
	department_head VARCHAR(50),
	department_location VARCHAR(50),
	department_creation_date DATE
);
"""

In [21]:
employee_schema = """
CREATE TABLE IF NOT EXISTS employee (
	employee_id INT PRIMARY KEY,
	first_name VARCHAR(50),
	last_name VARCHAR(50),
	age INT,
	email VARCHAR(50),
	gender VARCHAR(50),
	job_title VARCHAR(50),
	department VARCHAR(50),
	salary DECIMAL(8,2),
	hire_date DATE,
    FOREIGN KEY (department) REFERENCES department(department_id)
);
"""

In [22]:
db_name = 'employees.db'

if os.path.exists(db_name):
    os.remove(db_name)
    print(f"Removed existing database '{db_name}'.")

Removed existing database 'employees.db'.


In [23]:
pd.read_csv('employees.csv').head()

,employee_id,first_name,last_name,age,email,gender,job_title,department,salary,hire_date
0,1,Sara,Riccardini,68,sriccardini0@flickr.com,Female,Environmental Tech,478,132668.57,12/1/2022
1,2,Belia,Lang,51,blang1@blogtalkradio.com,Female,Pharmacist,165,116577.76,1/31/2012
2,3,Chevy,Halesworth,63,chalesworth2@toplist.cz,Male,Technical Writer,44,38419.86,6/20/2017
3,4,Miles,Pettecrew,84,mpettecrew3@springer.com,Male,Software Consultant,427,75318.66,1/28/2016
4,5,Martica,Le Huquet,48,mlehuquet4@stumbleupon.com,Female,Analog Circuit Design manager,309,49551.34,6/5/2013


In [24]:
COLUMN_DATA_TYPES = {
    'department': {
        'department_id': 'int64',
        'department_name':'object',
        'department_code': 'object',
        'department_head': 'object',
        'department_location': 'object',
        'department_creation_date': 'datetime64[ns]', 
    },
    'employee': {
        'employee_id': 'int64',
        'first_name': 'object',
        'last_name': 'object',
        'age': 'int64',
        'email': 'object',
        'gender' : 'object',
        'job_title': 'object',
        'department': 'int64',
        'salary': 'float64',
        'hire_date': 'datetime64[ns]'
                },
}



### Dataabase connection

In [25]:
conn = None  # Initialize connection to None

try:
    # Establish a connection to the SQLite database
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()
    print(f"Database '{db_name}' created and connected successfully. ✅")

    # Create tables
    cursor.execute(department_schema)
    cursor.execute(employee_schema)
    print("Tables created successfully.")


    # --- Load data from CSV files into the tables using pandas ---
    csv_to_table_map = {
        './Datasets/department.csv' : 'department',
        './Datasets/Employees.csv': 'employee',
    }

    for csv_file, table_name in csv_to_table_map.items():
        if os.path.exists(csv_file):
            print(f"\nProcessing '{csv_file}' for table '{table_name}'...")

            # Read the CSV file into a pandas DataFrame
            df = pd.read_csv(csv_file)

            # 1. Get the expected schema for the current table
            expected_schema = COLUMN_DATA_TYPES[table_name]
            expected_cols = list(expected_schema.keys())

            # 2. Handle missing/extra columns
            # Drop columns from DataFrame that are not in the schema
            df = df[df.columns.intersection(expected_cols)]

            # Add any missing columns and fill with None (which becomes NULL in SQL)
            for col in expected_cols:
                if col not in df.columns:
                    df[col] = None

            # 3. Reorder columns to match the defined schema exactly
            df = df[expected_cols]

            # 4. Enforce data types
            for col, dtype in expected_schema.items():
                if 'datetime' in dtype:
                    # Use pd.to_datetime for date/time columns, coercing errors to NaT (Not a Time)
                    df[col] = pd.to_datetime(df[col], errors='coerce')
                else:
                    # Use astype for other columns, handling potential conversion errors
                    try:
                        df[col] = df[col].astype(dtype)
                    except (ValueError, TypeError) as e:
                        print(f"  - Warning: Could not convert column '{col}' to {dtype}. Error: {e}. Leaving as is.")


            # Use the to_sql method to insert the cleaned DataFrame
            df.to_sql(table_name, conn, if_exists='append', index=False)
            print(f"  -> Data from '{csv_file}' loaded into '{table_name}' table successfully.")
        else:
            print(f"Warning: '{csv_file}' not found. Skipping data load for '{table_name}'.")
        
        # Commit the changes to the database
    conn.commit()
    print("\nData committed to the database successfully. 🎉")

except sqlite3.Error as e:
    print(f"Database error: {e}")
except pd.errors.EmptyDataError as e:
    print(f"Pandas error: {e}. One of the CSV files might be empty.")
except KeyError as e:
    print(f"Schema definition error: A column is missing from the TABLE_DATA_TYPES dictionary: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    # Close the connection if it was established
    if conn:
        conn.close()
        print("Database connection closed.")

Database 'employees.db' created and connected successfully. ✅
Tables created successfully.

Processing './Datasets/department.csv' for table 'department'...
  -> Data from './Datasets/department.csv' loaded into 'department' table successfully.

Processing './Datasets/Employees.csv' for table 'employee'...
  -> Data from './Datasets/Employees.csv' loaded into 'employee' table successfully.

Data committed to the database successfully. 🎉
Database connection closed.


### Install GEN AI Library

In [26]:
!pip install google-genai


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from dotenv import load_dotenv
from google import genai

load_dotenv('.env.local')

True

In [28]:
genai_client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))


### Prompts

In [29]:
pd.read_csv('employees.csv').head()

,employee_id,first_name,last_name,age,email,gender,job_title,department,salary,hire_date
0,1,Sara,Riccardini,68,sriccardini0@flickr.com,Female,Environmental Tech,478,132668.57,12/1/2022
1,2,Belia,Lang,51,blang1@blogtalkradio.com,Female,Pharmacist,165,116577.76,1/31/2012
2,3,Chevy,Halesworth,63,chalesworth2@toplist.cz,Male,Technical Writer,44,38419.86,6/20/2017
3,4,Miles,Pettecrew,84,mpettecrew3@springer.com,Male,Software Consultant,427,75318.66,1/28/2016
4,5,Martica,Le Huquet,48,mlehuquet4@stumbleupon.com,Female,Analog Circuit Design manager,309,49551.34,6/5/2013


In [30]:
prompt = """

###ROLE###
You are a highly skilled Text-to-SQL translator with expertise in SQL syntax, database schema interpretation, and natural language understanding. You generate syntactically correct and semantically accurate SQL queries based on user input and a given database schema.

###CONTEXT###
The user is working with a relational database for a company.. The database includes two main tables: `employees`,  and `department`. The goal is to allow users to input natural language queries (in English), and have the model return equivalent SQL statements that accurately extract the requested data using the given schema.

Here is the full schema:

**Employees Table**
```sql
CREATE TABLE IF NOT EXISTS employee (
	employee_id INT PRIMARY KEY,
	first_name VARCHAR(50),
	last_name VARCHAR(50),
	age INT,
	email VARCHAR(50),
	gender VARCHAR(50),
	job_title VARCHAR(50),
	department VARCHAR(50),
	salary DECIMAL(8,2),
	hire_date DATE,
    FOREIGN KEY (department) REFERENCES department(department_id)
);
````

*Department Table**

```sql
CREATE TABLE IF NOT EXISTS department (
	department_id INT PRIMARY KEY,
	department_name VARCHAR(50),
	department_code VARCHAR(9),
	department_head VARCHAR(50),
	department_location VARCHAR(50),
	department_creation_date DATE
);
```

###TASK###
Your task is to:

1. Read a natural language query about the data.
2. Interpret the user's intent based on the schema provided.
3. Generate a valid SQL `SELECT` query that returns the expected result.
4. Ensure correct table joins, column selection, filtering, and grouping as necessary.
5. Handle aggregate functions (e.g., `COUNT`, `AVG`, `SUM`) where appropriate.

###CONSTRAINTS###

* Only return a valid SQL query as output — no explanations or extra text.
* The user is using sqllite database - respond with correct and valid sqllite syntax
* Use aliases (`AS`) for column names only when the original name is ambiguous.
* Do not create or modify tables.
* Do not assume the existence of tables or columns not provided in the schema.
* Avoid subqueries unless absolutely necessary for correctness or performance.
* Prefer readability: indent joins and clauses properly.

###EXAMPLES###
**Input:** "Show me the names and emails of employees who is Professor."
**Output:**

```sql
SELECT first_name, last_name, email
FROM employee
WHERE job_title = 'Professor';
```

###OUTPUT FORMAT###
Return only the sqllite SQL query as a code block using triple backticks and the `sql` language tag, like this:

```sql
-- Your SQL query here
```
"""

In [32]:
import json
def get_sql_query_via_gemini(genai_client, prompt, user_query):

  # https://www.geeksforgeeks.org/python/formatted-string-literals-f-strings-python/
  contents = f"""
  {prompt}

  Here's the user query in english you need to work on:
  {user_query}
  """
  response = genai_client.models.generate_content(model='gemini-2.5-flash', contents=contents)
  # print(response) # uncomment this and understand at the output

  # Access the usage_metadata attribute
  usage_metadata = response.usage_metadata

  # Print the different token counts
  print(f"Input Token Count: {usage_metadata.prompt_token_count}")
  print(f"Thoughts Token Count: {response.usage_metadata.thoughts_token_count}")
  print(f"Output Token Count: {usage_metadata.candidates_token_count}")
  print(f"Total Token Count: {usage_metadata.total_token_count}")

  output = response.text.replace('```sql', '').replace('```', '')

  return output


In [34]:
import sqlite3
import pandas as pd

def execute_query(query, db_name='employees.db'):

    conn = None
    try:
        # Connect to the database
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Execute the query
        print(f"\nExecuting query on '{db_name}':\n{query}")
        cursor.execute(query)

        # Fetch all results
        results = cursor.fetchall()

        # Get column names from the cursor description
        columns = [description[0] for description in cursor.description]

        # Format results as a dataframe for easier use
        results_as_dict = [dict(zip(columns, row)) for row in results]
        results_df = pd.DataFrame(results_as_dict)

        print("Query executed successfully.")
        return results_df

    except sqlite3.Error as e:
        print(f"Database error executing query: {e}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None
    finally:
        if conn:
            conn.close()

In [35]:
def text2sql(genai_client, prompt, user_query):
  output = get_sql_query_via_gemini(genai_client, prompt, user_query)
  results = execute_query(output)
  return results

In [37]:
text2sql(genai_client, prompt, "What is the max age in each department")

Input Token Count: 655
Thoughts Token Count: 61
Output Token Count: 24
Total Token Count: 740

Executing query on 'employees.db':

SELECT department, MAX(age) AS max_age
FROM employee
GROUP BY department;

Query executed successfully.


,department,max_age
0,1,77
1,10,48
2,101,85
3,102,83
4,103,83
...,...,...
695,993,21
696,994,73
697,995,60
698,996,53
